# 🪣 Notebook 1: Token Bucket

**Token bucket** = a bucket that refills at a steady rate `r` tokens/second up to a cap `B`. Each request consumes one token. If the bucket is empty, the request is rejected (or queued).

It allows **bursts up to `B`** while keeping the long-term average ≤ `r`. Used by AWS, GCP, Stripe, NGINX, etc.


## 🛠️ Setup

```bash
cd 04-patterns/rate-limiting-and-throttling
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟩 Implementation

In [ ]:
import time

class TokenBucket:
    def __init__(self, rate, capacity):
        self.rate = rate
        self.capacity = capacity
        self.tokens = capacity
        self.last = time.monotonic()

    def allow(self, n=1):
        now = time.monotonic()
        # Refill based on elapsed time
        self.tokens = min(self.capacity, self.tokens + (now - self.last) * self.rate)
        self.last = now
        if self.tokens >= n:
            self.tokens -= n
            return True
        return False


## 💥 Let a burst of 20 hit a 5/s limiter (capacity=10)

In [ ]:
tb = TokenBucket(rate=5, capacity=10)
results = [tb.allow() for _ in range(20)]
print('burst result (1=allow, 0=deny):', [int(r) for r in results])
print(f'allowed {sum(results)} of {len(results)} — bucket emptied after the burst')

time.sleep(1.0)
print('after 1s wait:', [int(tb.allow()) for _ in range(8)])


Bursts up to `B=10` are allowed; after that we wait for the bucket to refill.